app.py

In [ ]:
from flask import Flask, render_template, request, jsonify
from services.speech_to_text import transcribe_audio
import threading
import os

app = Flask(__name__)

progress = 0
transcript_result = ""
sentiment_result = ""
is_processing = False


def process_audio(path):
    global progress, transcript_result, sentiment_result, is_processing

    try:
        progress = 10

        transcript = transcribe_audio(path, update_progress)

        transcript_result = transcript
        sentiment_result = "Positive"  # nanti bisa diganti AI

        progress = 100

    except Exception as e:
        transcript_result = f"Error: {str(e)}"
        progress = 0

    is_processing = False


@app.route("/", methods=["GET", "POST"])
def index():
    global is_processing

    if request.method == "POST":

        if is_processing:
            return "Masih memproses audio sebelumnya..."

        audio = request.files["audio"]

        if audio.filename == "":
            return "File kosong"

        os.makedirs("uploads", exist_ok=True)

        path = "uploads/audio.wav"
        audio.save(path)

        is_processing = True

        # jalankan di background
        thread = threading.Thread(target=process_audio, args=(path,))
        thread.start()

    return render_template("index.html",
        transcript=transcript_result,
        sentiment=sentiment_result
    )


@app.route("/progress")
def get_progress():
    return jsonify({
        "progress": progress,
        "processing": is_processing
    })


def update_progress(value):
    global progress
    progress = min(value, 100)


if __name__ == "__main__":
    app.run(debug=True)

app.py 28/april


In [ ]:
from flask import Flask, render_template, request, jsonify, send_file
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip
from faster_whisper import WhisperModel

import threading
import os
import subprocess
import difflib
import pandas as pd

app = Flask(__name__)

# 🔥 MODEL (pakai parameter optimal)
model = WhisperModel("medium", compute_type="int8")

# GLOBAL
progress = 0
transcript_result = ""
ground_truth_result = ""
wer_result = "-"
cer_result = "-"
accuracy_result = "-"
highlight_result = ""
is_processing = False

# 🔥 timeline
time_labels = []
accuracy_timeline = []
wer_timeline = []

history = []

# NORMALIZER
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    RemoveMultipleSpaces(),
    Strip()
])


def convert_audio(input_path, output_path):
    subprocess.run([
        "ffmpeg", "-y",
        "-i", input_path,
        "-ar", "16000",
        "-ac", "1",
        output_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


# 🔥 highlight error
def highlight_diff(ref, hyp):
    ref_words = ref.split()
    hyp_words = hyp.split()

    diff = difflib.ndiff(ref_words, hyp_words)

    result = []
    for d in diff:
        if d.startswith("- "):
            result.append(f"<span style='color:red'>{d[2:]}</span>")
        elif d.startswith("+ "):
            result.append(f"<span style='color:green'>{d[2:]}</span>")
        else:
            result.append(d[2:])

    return " ".join(result)


def process_audio(path, ground_truth):
    global progress, transcript_result, ground_truth_result
    global wer_result, cer_result, accuracy_result, highlight_result, is_processing
    global time_labels, accuracy_timeline, wer_timeline

    try:
        progress = 10

        convert_audio(path, "uploads/clean.wav")
        progress = 30

        # 🔥 TRANSCRIBE (SUDAH BAGUS)
        segments, _ = model.transcribe(
            "uploads/clean.wav",
            language="id",
            beam_size=3,
            vad_filter=True
        )

        segments = list(segments)

        full_text = ""
        time_labels.clear()
        accuracy_timeline.clear()
        wer_timeline.clear()

        # 🔥 gabungkan transcript
        for seg in segments:
            full_text += seg.text.strip() + " "

        transcript = full_text.strip()
        transcript_result = transcript
        ground_truth_result = ground_truth

        progress = 70

        # =========================
        # 🔥 FINAL EVALUATION (VALID)
        # =========================
        if ground_truth.strip():
            ref = transform(ground_truth)
            hyp = transform(transcript)

            w = wer(ref, hyp)
            c = cer(ref, hyp)
            acc = (1 - w) * 100

            wer_result = round(w, 4)
            cer_result = round(c, 4)
            accuracy_result = round(acc, 2)

            highlight_result = highlight_diff(ref, hyp)

            history.append({
                "WER": wer_result,
                "CER": cer_result,
                "Accuracy": accuracy_result
            })

        else:
            wer_result = "-"
            cer_result = "-"
            accuracy_result = "-"
            highlight_result = ""

        # =========================
        # 🔥 TIMELINE (FIX VALID)
        # =========================
        if ground_truth.strip():

            gt_clean = transform(ground_truth)
            hyp_running = ""

            for seg in segments:
                seg_text = seg.text.strip()

                if not seg_text:
                    continue

                # 🔥 bangun transcript bertahap
                hyp_running += " " + seg_text
                hyp_clean = transform(hyp_running)

                # 🔥 PROPORSIONAL ALIGNMENT
                ratio = len(hyp_clean) / max(len(gt_clean), 1)
                cut_index = int(len(gt_clean) * ratio)

                gt_slice = gt_clean[:cut_index]

                if len(gt_slice) > 5:
                    w = wer(gt_slice, hyp_clean)
                    acc = (1 - w) * 100

                    time_labels.append(f"{round(seg.end,1)}s")
                    accuracy_timeline.append(round(acc, 2))
                    wer_timeline.append(round(w, 3))

        progress = 100

    except Exception as e:
        transcript_result = f"Error: {str(e)}"
        progress = 0

    is_processing = False


@app.route("/", methods=["GET", "POST"])
def index():
    global is_processing

    if request.method == "POST":

        if is_processing:
            return jsonify({"status": "processing"})

        audio = request.files.get("audio")
        gt_file = request.files.get("ground_truth")

        os.makedirs("uploads", exist_ok=True)

        raw_path = os.path.join("uploads", audio.filename)
        audio.save(raw_path)

        ground_truth_text = ""
        if gt_file:
            ground_truth_text = gt_file.read().decode("utf-8", errors="ignore")

        is_processing = True

        thread = threading.Thread(
            target=process_audio,
            args=(raw_path, ground_truth_text)
        )
        thread.start()

        return jsonify({"status": "started"})

    return render_template("index.html")


@app.route("/progress")
def get_progress():
    return jsonify({
        "progress": progress,
        "processing": is_processing,
        "transcript": transcript_result,
        "ground_truth": ground_truth_result,
        "wer": wer_result,
        "cer": cer_result,
        "accuracy": accuracy_result,
        "highlight": highlight_result,
        "history": history,
        "time_labels": time_labels,
        "accuracy_timeline": accuracy_timeline,
        "wer_timeline": wer_timeline
    })


@app.route("/export")
def export():
    df = pd.DataFrame(history)
    file_path = "evaluation_result.xlsx"
    df.to_excel(file_path, index=False)
    return send_file(file_path, as_attachment=True)


def update_progress(value):
    global progress
    progress = min(value, 100)


if __name__ == "__main__":
    app.run(debug=True)

In [ ]:
from flask import Flask, render_template, request, jsonify, send_file
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip
from faster_whisper import WhisperModel

import threading
import os
import subprocess
import difflib
import pandas as pd

app = Flask(__name__)

# 🔥 MODEL (pakai parameter optimal)
model = WhisperModel("medium", compute_type="int8")

# GLOBAL
progress = 0
transcript_result = ""
ground_truth_result = ""
wer_result = "-"
cer_result = "-"
accuracy_result = "-"
highlight_result = ""
is_processing = False

# 🔥 timeline
time_labels = []
accuracy_timeline = []
wer_timeline = []

history = []

# NORMALIZER
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    RemoveMultipleSpaces(),
    Strip()
])


def convert_audio(input_path, output_path):
    subprocess.run([
        "ffmpeg", "-y",
        "-i", input_path,
        "-ar", "16000",
        "-ac", "1",
        output_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


# 🔥 highlight error
def highlight_diff(ref, hyp):
    ref_words = ref.split()
    hyp_words = hyp.split()

    diff = difflib.ndiff(ref_words, hyp_words)

    result = []
    for d in diff:
        if d.startswith("- "):
            result.append(f"<span style='color:red'>{d[2:]}</span>")
        elif d.startswith("+ "):
            result.append(f"<span style='color:green'>{d[2:]}</span>")
        else:
            result.append(d[2:])

    return " ".join(result)


def process_audio(path, ground_truth):
    global progress, transcript_result, ground_truth_result
    global wer_result, cer_result, accuracy_result, highlight_result, is_processing
    global time_labels, accuracy_timeline, wer_timeline

    try:
        progress = 10

        convert_audio(path, "uploads/clean.wav")
        progress = 30

        # 🔥 TRANSCRIBE (FIX UTAMA DI SINI)
        segments, _ = model.transcribe(
            "uploads/clean.wav",
            language="id",
            beam_size=3,          # 🔥 penting (akurasi naik)
            vad_filter=True       # 🔥 penting (hilangkan noise)
        )

        segments = list(segments)

        full_text = ""
        time_labels.clear()
        accuracy_timeline.clear()
        wer_timeline.clear()

        # 🔥 gabungkan text
        for seg in segments:
            full_text += seg.text.strip() + " "

        transcript = full_text.strip()
        transcript_result = transcript
        ground_truth_result = ground_truth

        progress = 70

        # 🔥 EVALUASI FINAL (VALID)
        if ground_truth.strip():
            ref = transform(ground_truth)
            hyp = transform(transcript)

            w = wer(ref, hyp)
            c = cer(ref, hyp)
            acc = (1 - w) * 100

            wer_result = round(w, 4)
            cer_result = round(c, 4)
            accuracy_result = round(acc, 2)

            highlight_result = highlight_diff(ref, hyp)

            history.append({
                "WER": wer_result,
                "CER": cer_result,
                "Accuracy": accuracy_result
            })

        else:
            wer_result = "-"
            cer_result = "-"
            accuracy_result = "-"
            highlight_result = ""

        # 🔥 TIMELINE (STABIL — TANPA slicing aneh)
        if ground_truth.strip():
            for seg in segments:
                seg_text = seg.text.strip()

                if not seg_text:
                    continue

                ref = transform(ground_truth)
                hyp = transform(seg_text)

                w = wer(ref, hyp)
                acc = (1 - w) * 100

                time_labels.append(f"{round(seg.end,1)}s")
                accuracy_timeline.append(round(acc, 2))
                wer_timeline.append(round(w, 3))

        progress = 100

    except Exception as e:
        transcript_result = f"Error: {str(e)}"
        progress = 0

    is_processing = False


@app.route("/", methods=["GET", "POST"])
def index():
    global is_processing

    if request.method == "POST":

        if is_processing:
            return jsonify({"status": "processing"})

        audio = request.files.get("audio")
        gt_file = request.files.get("ground_truth")

        os.makedirs("uploads", exist_ok=True)

        raw_path = os.path.join("uploads", audio.filename)
        audio.save(raw_path)

        ground_truth_text = ""
        if gt_file:
            ground_truth_text = gt_file.read().decode("utf-8", errors="ignore")

        is_processing = True

        thread = threading.Thread(
            target=process_audio,
            args=(raw_path, ground_truth_text)
        )
        thread.start()

        return jsonify({"status": "started"})

    return render_template("index.html")


@app.route("/progress")
def get_progress():
    return jsonify({
        "progress": progress,
        "processing": is_processing,
        "transcript": transcript_result,
        "ground_truth": ground_truth_result,
        "wer": wer_result,
        "cer": cer_result,
        "accuracy": accuracy_result,
        "highlight": highlight_result,
        "history": history,
        "time_labels": time_labels,
        "accuracy_timeline": accuracy_timeline,
        "wer_timeline": wer_timeline
    })


@app.route("/export")
def export():
    df = pd.DataFrame(history)
    file_path = "evaluation_result.xlsx"
    df.to_excel(file_path, index=False)
    return send_file(file_path, as_attachment=True)


def update_progress(value):
    global progress
    progress = min(value, 100)


if __name__ == "__main__":
    app.run(debug=True)

CODE FIX DIBAWAH

In [ ]:
from flask import Flask, render_template, request, jsonify, send_file
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip
from faster_whisper import WhisperModel

import threading
import os
import subprocess
import difflib
import pandas as pd

app = Flask(__name__)

# MODEL
model = WhisperModel("medium", compute_type="int8")

# GLOBAL
progress = 0
transcript_result = ""
ground_truth_result = ""
wer_result = "-"
cer_result = "-"
accuracy_result = "-"
highlight_result = ""
is_processing = False

time_labels = []
accuracy_timeline = []
wer_timeline = []
history = []

# NORMALIZER
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    RemoveMultipleSpaces(),
    Strip()
])

def normalize_text(text):
    return " ".join(text.lower().split())

# =========================
# AUDIO CONVERT
# =========================
def convert_audio(input_path, output_path):
    subprocess.run([
        "ffmpeg", "-y",
        "-i", input_path,
        "-ar", "16000",
        "-ac", "1",
        output_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# =========================
# HIGHLIGHT
# =========================
def highlight_diff(ref, hyp):
    ref_words = ref.split()
    hyp_words = hyp.split()

    diff = difflib.ndiff(ref_words, hyp_words)

    result = []
    for d in diff:
        if d.startswith("- "):
            result.append(f"<span style='color:red'>{d[2:]}</span>")
        elif d.startswith("+ "):
            result.append(f"<span style='color:green'>{d[2:]}</span>")
        else:
            result.append(d[2:])

    return " ".join(result)

# =========================
# MAIN PROCESS
# =========================
def process_audio(path, ground_truth):
    global progress, transcript_result, ground_truth_result
    global wer_result, cer_result, accuracy_result, highlight_result, is_processing
    global time_labels, accuracy_timeline, wer_timeline

    try:
        progress = 10

        convert_audio(path, "uploads/clean.wav")
        progress = 30

        # 🔥 TRANSCRIBE (VERSI LAMA - AKURASI TINGGI)
        segments, _ = model.transcribe(
            "uploads/clean.wav",
            language="id",
            beam_size=3,
            vad_filter=True
        )

        segments = list(segments)

        full_text = ""
        time_labels.clear()
        accuracy_timeline.clear()
        wer_timeline.clear()

        # =========================
        # 🔥 BUILD FULL TRANSCRIPT
        # =========================
        for seg in segments:
            full_text += seg.text.strip() + " "

        transcript = full_text.strip()
        transcript_result = transcript
        ground_truth_result = ground_truth

        progress = 70

        # =========================
        # 🔥 FINAL EVALUATION
        # =========================
        if ground_truth.strip():
            ref = normalize_text(transform(ground_truth))
            hyp = normalize_text(transform(transcript))

            w = wer(ref, hyp)
            c = cer(ref, hyp)
            acc = (1 - w) * 100

            wer_result = round(w, 4)
            cer_result = round(c, 4)
            accuracy_result = round(acc, 2)

            highlight_result = highlight_diff(ref, hyp)

            history.append({
                "WER": wer_result,
                "CER": cer_result,
                "Accuracy": accuracy_result
            })

        else:
            wer_result = "-"
            cer_result = "-"
            accuracy_result = "-"
            highlight_result = ""

        # =========================
        # 🔥 TIMELINE (VERSI BARU - VALID)
        # =========================
        if ground_truth.strip():

            gt_clean = normalize_text(transform(ground_truth))
            hyp_running = ""

            for seg in segments:
                seg_text = seg.text.strip()

                if not seg_text:
                    continue

                # 🔥 akumulatif
                hyp_running += " " + seg_text
                hyp_clean = normalize_text(transform(hyp_running))

                # 🔥 proporsional
                ratio = len(hyp_clean) / max(len(gt_clean), 1)
                cut_index = int(len(gt_clean) * ratio)

                gt_slice = gt_clean[:cut_index]

                if len(gt_slice) > 10:
                    w = wer(gt_slice, hyp_clean)
                    acc = (1 - w) * 100

                    time_labels.append(f"{round(seg.end,1)}s")
                    accuracy_timeline.append(round(acc, 2))
                    wer_timeline.append(round(w, 3))

        progress = 100

    except Exception as e:
        transcript_result = f"Error: {str(e)}"
        progress = 0

    is_processing = False

# =========================
# ROUTES
# =========================
@app.route("/", methods=["GET", "POST"])
def index():
    global is_processing

    if request.method == "POST":

        if is_processing:
            return jsonify({"status": "processing"})

        audio = request.files.get("audio")
        gt_file = request.files.get("ground_truth")

        os.makedirs("uploads", exist_ok=True)

        raw_path = os.path.join("uploads", audio.filename)
        audio.save(raw_path)

        ground_truth_text = ""
        if gt_file:
            ground_truth_text = gt_file.read().decode("utf-8", errors="ignore")

        is_processing = True

        thread = threading.Thread(
            target=process_audio,
            args=(raw_path, ground_truth_text)
        )
        thread.start()

        return jsonify({"status": "started"})

    return render_template("index.html")

@app.route("/progress")
def get_progress():
    return jsonify({
        "progress": progress,
        "processing": is_processing,
        "transcript": transcript_result,
        "ground_truth": ground_truth_result,
        "wer": wer_result,
        "cer": cer_result,
        "accuracy": accuracy_result,
        "highlight": highlight_result,
        "history": history,
        "time_labels": time_labels,
        "accuracy_timeline": accuracy_timeline,
        "wer_timeline": wer_timeline
    })

@app.route("/export")
def export():
    df = pd.DataFrame(history)
    file_path = "evaluation_result.xlsx"
    df.to_excel(file_path, index=False)
    return send_file(file_path, as_attachment=True)

def update_progress(value):
    global progress
    progress = min(value, 100)

if __name__ == "__main__":
    app.run(debug=True)